# M-Shots Learning


In this lab, we'll explore zero-shot, one-shot, and few-shot prompting. We'll compare how examples and explicit instructions influence the consistency and format of model responses.


## Setup

Install the dependencies if needed (`pip install -U openai python-dotenv`). Create a `.env` file containing `OPENAI_API_KEY=your_api_key`, or set the environment variable in your shell. Never commit your API key to version control.


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from getpass import getpass

load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"] = getpass("Enter API key: ")

client = OpenAI()
MODEL = "gpt-4o-mini"


Enter API key:  ········


## Few-shot prompting

A *shot* is an example included in a prompt:

- **Zero-shot:** no examples; the model relies on the instructions and its prior training.
- **One-shot:** one example.
- **Few-shot:** multiple examples.

Examples can help a model follow a desired style, label set, or output format. There is no universally correct number of examples: results depend on the task, the examples, and the model. Examples also consume input tokens, so use a small, representative set and evaluate the results.

This notebook uses OpenAI's **Responses API** with `gpt-4o-mini`. The API accepts a list of role-based messages as `input`; the response text is available through `response.output_text`.


In [2]:
def return_OAIResponse(user_message, context):
    """Send role-based prompt context and a question to the Responses API."""
    messages = [*context, {"role": "user", "content": f"Question: {user_message}"}]
    response = client.responses.create(
        model=MODEL,
        input=messages,
    )
    return response.output_text


## 1. Zero-shot prompting

With no examples, the model may answer correctly but choose its own format. Inspect the output and note which details are present or missing.


In [3]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer factual questions clearly."}
]

In [4]:
# let's see the result with zero-shot:
print(return_OAIResponse("Who won the 2010 F1 Drivers' Championship?", context_user))

Sebastian Vettel won the 2010 F1 Drivers' Championship. He drove for Red Bull Racing and became the youngest World Champion in Formula One history at that time.


## 2. One-shot prompting

Now include one example as a user/assistant exchange. The example demonstrates the format we want; it does not guarantee that every answer will be correct.


In [5]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2000 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Michael Schumacher\nTeam: Ferrari\nPoints: 108"},
]

In [6]:
# let's see the result with one-shot:
print(return_OAIResponse("Who won the 2011 F1 Drivers' Championship?", context_user))

Driver: Sebastian Vettel  
Team: Red Bull Racing  
Points: 392  


## 3. Few-shot prompting

Multiple examples show the format across more than one case. Keep examples accurate and representative of the task.


In [7]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2010 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Sebastian Vettel\nTeam: Red Bull Racing\nPoints: 256"},
    {"role": "user", "content": "Who won the 2009 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Jenson Button\nTeam: Brawn GP\nPoints: 95"},
]
print(return_OAIResponse("Who won the 2006 F1 Drivers' Championship?", context_user))


Driver: Fernando Alonso  
Team: Renault  
Points: 134  


In [8]:
# let's see the result with few-shot:
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))

Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413  


## 4. Examples as messages versus instructions

Role-based examples make the demonstration clearer: each example question is a `user` message and its expected answer is an `assistant` message. The system message states the task. You can compare this with a prompt that explains the format directly. Neither approach is always best; test both on the same questions.


In [9]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer with exactly these fields, each on its own line: Driver, Team, Points. Use the championship winner's season-end points."}
]
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))


Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413


## 5. Classification with few-shot examples

Few-shot prompting can also be useful for other types of tasks, including classification. By providing examples of inputs and their expected labels, we can guide a model toward the desired behavior.

In this example, we'll classify product reviews by sentiment into three categories: **Positive**, **Negative**, or **Neutral**. Notice how the examples illustrate both the task and the expected output format.


In [10]:
context_user = [
    {"role": "system", "content": "Classify the product review as Positive, Negative, or Neutral. Return only the label."},
    {"role": "user", "content": "It fulfilled its function perfectly, the price is fair, and I would buy it again."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It did not work well and I would not buy it again. It is too expensive."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "I do not know what to say; my son uses it, but he does not love it."},
    {"role": "assistant", "content": "Neutral"},
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))


Neutral


## Exercise

- Choose a task (for example, extracting fields, classifying feedback, or rewriting text).
    - For example: Extract the product name, price, and category from a product description.
- Create zero-shot, one-shot, and few-shot versions of your prompt.
- Test each version on the same set of at least five inputs.
- Compare correctness, consistency, and adherence to the requested format. Include any failures or hallucinations you observe.
- Write a one-page report summarizing your findings and what you learned.


## Exercise

Your task is to extract structured information from product descriptions.

Use the following product description as your starting example:

> The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.

1. Create zero-shot, one-shot, and few-shot prompts to extract the following fields:
   - **Product name**
   - **Price**
   - **Category**

2. Generate three additional product descriptions (using code or an LLM) and test all three prompts on the same descriptions.

3. Compare the results in terms of accuracy, consistency, and adherence to the expected output format. Include any errors or hallucinations you observe.

4. Write a brief, simple report summarizing your findings and what you learned about the impact of examples on model performance.

## Notes

- Model output can vary and may contain factual errors; verify factual answers independently.
- Remember to keep API keys private. API usage may incur charges.


## Completed solution — Product information extraction

We evaluate the same **five descriptions** with zero-shot, one-shot and few-shot prompting. Expected values are manually defined, so accuracy can be measured without relying on another model.

In [11]:
import json
import pandas as pd

# Five evaluation cases, including missing values and an ambiguous price.
test_cases = [
    {"description": "The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.",
     "expected": {"product_name": "SoundWave Pro", "price": "€89.99", "category": "Electronics"}},
    {"description": "The UrbanTrail Backpack has a 25-liter capacity and costs €49.50. Category: Accessories.",
     "expected": {"product_name": "UrbanTrail Backpack", "price": "€49.50", "category": "Accessories"}},
    {"description": "FreshBrew Coffee Maker is a compact kitchen appliance priced at €79.00 in the Home Appliances category.",
     "expected": {"product_name": "FreshBrew Coffee Maker", "price": "€79.00", "category": "Home Appliances"}},
    {"description": "The FlexFit Yoga Mat, category Sports, is available for €24.99 in blue or green.",
     "expected": {"product_name": "FlexFit Yoga Mat", "price": "€24.99", "category": "Sports"}},
    {"description": "The LunaGlow Desk Lamp belongs to the Lighting category. Contact us for pricing.",
     "expected": {"product_name": "LunaGlow Desk Lamp", "price": None, "category": "Lighting"}},
]

# Common instructions keep the output schema comparable across methods.
schema_instruction = ("Extract product_name, price, and category from the description. "
    "Return ONLY a JSON object with exactly these three keys. "
    "Keep price as the original currency string. Use null when a field is absent. "
    "Never guess missing information.")

example_1 = ("The AeroLite Running Shoes cost €64.90 and are in the Footwear category.",
             '{"product_name":"AeroLite Running Shoes","price":"€64.90","category":"Footwear"}')
example_2 = ("The NovaSmart Watch is in Electronics. Price not listed.",
             '{"product_name":"NovaSmart Watch","price":null,"category":"Electronics"}')
example_3 = ("The CozyNest Blanket is €35.00. Category: Home Textiles.",
             '{"product_name":"CozyNest Blanket","price":"€35.00","category":"Home Textiles"}')

def build_messages(description, mode):
    messages = [{"role": "system", "content": schema_instruction}]
    examples = [] if mode == "zero-shot" else [example_1] if mode == "one-shot" else [example_1, example_2, example_3]
    for question, answer in examples:
        messages += [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    messages.append({"role": "user", "content": description})
    return messages

# Preview one prompt without calling the API.
print(json.dumps(build_messages(test_cases[0]["description"], "few-shot"), indent=2))


[
  {
    "role": "system",
    "content": "Extract product_name, price, and category from the description. Return ONLY a JSON object with exactly these three keys. Keep price as the original currency string. Use null when a field is absent. Never guess missing information."
  },
  {
    "role": "user",
    "content": "The AeroLite Running Shoes cost \u20ac64.90 and are in the Footwear category."
  },
  {
    "role": "assistant",
    "content": "{\"product_name\":\"AeroLite Running Shoes\",\"price\":\"\u20ac64.90\",\"category\":\"Footwear\"}"
  },
  {
    "role": "user",
    "content": "The NovaSmart Watch is in Electronics. Price not listed."
  },
  {
    "role": "assistant",
    "content": "{\"product_name\":\"NovaSmart Watch\",\"price\":null,\"category\":\"Electronics\"}"
  },
  {
    "role": "user",
    "content": "The CozyNest Blanket is \u20ac35.00. Category: Home Textiles."
  },
  {
    "role": "assistant",
    "content": "{\"product_name\":\"CozyNest Blanket\",\"price\":\"\u20a

### Run the three prompting strategies

Requires a valid `OPENAI_API_KEY`. Each call may incur API charges. The experiment makes 15 calls.

In [12]:
def extract_product(description, mode):
    response = client.responses.create(
        model=MODEL,
        input=build_messages(description, mode),
        temperature=0,  # Reduce variation where supported by the selected model.
    )
    return response.output_text.strip()

results = []
for mode in ["zero-shot", "one-shot", "few-shot"]:
    for case_id, case in enumerate(test_cases, start=1):
        try:
            raw = extract_product(case["description"], mode)
            try:
                parsed = json.loads(raw)
            except json.JSONDecodeError:
                parsed = None
            results.append({"mode": mode, "case": case_id, "raw_output": raw,
                            "parsed": parsed, "expected": case["expected"], "error": None})
        except Exception as exc:
            results.append({"mode": mode, "case": case_id, "raw_output": "",
                            "parsed": None, "expected": case["expected"], "error": str(exc)})

for row in results:
    print(f"{row['mode']:10} | Case {row['case']}: {row['raw_output'] or row['error']}")


zero-shot  | Case 1: ```json
{
  "product_name": "SoundWave Pro wireless headphones",
  "price": "€89.99",
  "category": "Electronics"
}
```
zero-shot  | Case 2: ```json
{
  "product_name": "UrbanTrail Backpack",
  "price": "€49.50",
  "category": "Accessories"
}
```
zero-shot  | Case 3: ```json
{
  "product_name": "FreshBrew Coffee Maker",
  "price": "€79.00",
  "category": "Home Appliances"
}
```
zero-shot  | Case 4: ```json
{
  "product_name": "The FlexFit Yoga Mat",
  "price": "€24.99",
  "category": "Sports"
}
```
zero-shot  | Case 5: ```json
{
  "product_name": "LunaGlow Desk Lamp",
  "price": null,
  "category": "Lighting"
}
```
one-shot   | Case 1: {"product_name":"SoundWave Pro wireless headphones","price":"€89.99","category":"Electronics"}
one-shot   | Case 2: {"product_name":"UrbanTrail Backpack","price":"€49.50","category":"Accessories"}
one-shot   | Case 3: {"product_name":"FreshBrew Coffee Maker","price":"€79.00","category":"Home Appliances"}
one-shot   | Case 4: {"produc

### Evaluate correctness, formatting, and hallucinations

*Accuracy* is the proportion of expected field values extracted correctly. *Format adherence* requires valid JSON with exactly the three requested keys. A *hallucinated missing value* is a non-null output for a field whose reference value is null. Results are measured only after running the API calls.

In [13]:
required_keys = {"product_name", "price", "category"}
for row in results:
    parsed = row["parsed"]
    valid = isinstance(parsed, dict) and set(parsed) == required_keys
    row["valid_format"] = valid
    row["correct_fields"] = sum(parsed.get(k) == v for k, v in row["expected"].items()) if valid else 0
    row["field_accuracy"] = row["correct_fields"] / 3
    row["exact_match"] = valid and parsed == row["expected"]
    row["hallucinated_missing"] = (sum(v is None and parsed.get(k) is not None
                                   for k, v in row["expected"].items()) if valid else 0)

scores = pd.DataFrame(results)
summary = (scores.groupby("mode", sort=False)
           .agg(field_accuracy=("field_accuracy", "mean"),
                format_adherence=("valid_format", "mean"),
                exact_match_rate=("exact_match", "mean"),
                hallucinated_missing_values=("hallucinated_missing", "sum"),
                api_errors=("error", lambda s: s.notna().sum()))
           .reset_index())
display(summary)

# Inspect individual failures and discrepancies.
failures = scores[~scores["exact_match"]][["mode", "case", "raw_output", "expected", "error"]]
display(failures)


,mode,field_accuracy,format_adherence,exact_match_rate,hallucinated_missing_values,api_errors
0,zero-shot,0.000000,0.0,0.0,0,0
1,one-shot,0.933333,1.0,0.8,0,0
2,few-shot,0.933333,1.0,0.8,0,0


,mode,case,raw_output,expected,error
0,zero-shot,1,"```json\n{\n ""product_name"": ""SoundWave Pro w...","{'product_name': 'SoundWave Pro', 'price': '€8...",None
1,zero-shot,2,"```json\n{\n ""product_name"": ""UrbanTrail Back...","{'product_name': 'UrbanTrail Backpack', 'price...",None
2,zero-shot,3,"```json\n{\n ""product_name"": ""FreshBrew Coffe...","{'product_name': 'FreshBrew Coffee Maker', 'pr...",None
3,zero-shot,4,"```json\n{\n ""product_name"": ""The FlexFit Yog...","{'product_name': 'FlexFit Yoga Mat', 'price': ...",None
4,zero-shot,5,"```json\n{\n ""product_name"": ""LunaGlow Desk L...","{'product_name': 'LunaGlow Desk Lamp', 'price'...",None
5,one-shot,1,"{""product_name"":""SoundWave Pro wireless headph...","{'product_name': 'SoundWave Pro', 'price': '€8...",None
10,few-shot,1,"{""product_name"":""SoundWave Pro wireless headph...","{'product_name': 'SoundWave Pro', 'price': '€8...",None


### Generate a report from the actual experiment

The report is produced from measured results rather than invented scores.

In [14]:
def pct(value):
    return f"{value:.1%}"

lines = ["# M-Shots Prompting — Experiment Report", "",
         "## Objective", "Compare zero-shot, one-shot and few-shot extraction of product name, price and category.", "",
         "## Method", "Five fixed product descriptions were tested with each strategy using the same model and output instructions. "
         "One-shot uses one demonstration; few-shot uses three. Expected outputs were defined in advance.", "",
         "## Results"]
for _, r in summary.iterrows():
    lines.append(f"- {r['mode']}: field accuracy {pct(r['field_accuracy'])}; "
                 f"valid JSON/schema {pct(r['format_adherence'])}; "
                 f"exact matches {pct(r['exact_match_rate'])}; "
                 f"hallucinated missing values {int(r['hallucinated_missing_values'])}; "
                 f"API errors {int(r['api_errors'])}.")
lines += ["", "## Errors and hallucinations"]
if failures.empty:
    lines.append("No mismatches were observed in this run.")
else:
    for _, r in failures.iterrows():
        lines.append(f"- {r['mode']}, case {r['case']}: output={r['raw_output']!r}; expected={r['expected']}; error={r['error']}")
lines += ["", "## Conclusion", "Examples can clarify the intended extraction pattern, but do not guarantee improved accuracy. "
          "Because the schema is already explicit in all three prompts, differences may be small. "
          "The missing-price example tests whether the model invents unavailable data. "
          "Repeat the experiment with more descriptions and multiple runs before drawing broad conclusions."]
report = "\n".join(lines)
print(report)
with open("m_shots_report.md", "w", encoding="utf-8") as file:
    file.write(report)


# M-Shots Prompting — Experiment Report

## Objective
Compare zero-shot, one-shot and few-shot extraction of product name, price and category.

## Method
Five fixed product descriptions were tested with each strategy using the same model and output instructions. One-shot uses one demonstration; few-shot uses three. Expected outputs were defined in advance.

## Results
- zero-shot: field accuracy 0.0%; valid JSON/schema 0.0%; exact matches 0.0%; hallucinated missing values 0; API errors 0.
- one-shot: field accuracy 93.3%; valid JSON/schema 100.0%; exact matches 80.0%; hallucinated missing values 0; API errors 0.
- few-shot: field accuracy 93.3%; valid JSON/schema 100.0%; exact matches 80.0%; hallucinated missing values 0; API errors 0.

## Errors and hallucinations
- zero-shot, case 1: output='```json\n{\n  "product_name": "SoundWave Pro wireless headphones",\n  "price": "€89.99",\n  "category": "Electronics"\n}\n```'; expected={'product_name': 'SoundWave Pro', 'price': '€89.99', 'cate

## Additional exercise — Choose your own task

The earlier open-ended exercise is also satisfied by the same extraction task: five inputs, three prompt strategies, measured correctness and format adherence, failure inspection, and an automatically generated report. The specific product-extraction exercise uses the required SoundWave Pro starting example and four further test cases.